# First EUV experiment on Colab

This notebook is an experiment control panel. It calls the Python code in `src/`; it does not contain a separate optimizer.

**Flow:** notebook configuration → `run_optimization.py` → `bayesopt/optimization.py` → `srcmdl.py` → saved results → notebook display.

A **free CPU runtime** is enough for the baseline GP. To test GPU access too, select a GPU runtime and run the optional GPU check below. The existing NumPy/scikit-learn GP models still execute on CPU.

**Scope:** this is a small execution test with selectable GP or Kumo EUV prediction. Kumo can replace EUV only or all five response predictors via `SURROGATE_SCOPE`. Set `EUV_MODEL` below to choose. Its yield term is still a soft penalty, not a hard 99.5% constraint. A successful run does not mean that requirement has been implemented or validated.

## Before running cells

The default experiment uses **pretrained Kumo Large with four estimators**. No custom checkpoint upload is needed for this run.

**VS Code:** install Google's official **Colab** extension (and Jupyter if prompted). Open this notebook, choose **Select Kernel → Colab → Auto Connect**, and sign in. For a GPU access test, choose **New Colab Server** instead of Auto Connect and select an available free GPU (for example T4, if offered). For the CPU baseline alone, Auto Connect is sufficient. Then right-click the local **`src` folder → Upload to Colab**. Uploading the folder should create `/content/src`.

**Colab website alternative:** upload this `.ipynb` through File → Upload notebook. For a GPU test, choose **Runtime → Change runtime type → a free GPU option → Save**, then Connect. Upload the complete `src/` folder, including its `bayesopt/` subfolder, using Colab's Files pane. Preserve the folder structure; the setup cell accepts `/content/src` or `/content/ASML/src`.

For the fine-tuned run in VS Code, right-click the local **`data/checkpoint_upload_parts` folder → Upload to Colab**. This uploads 52 small parts (at most 16 MB each) and a manifest. The preparation cell joins them into `/content/data/lr1e-05_T1000_tcont128_tquery512_vcont256_vquery82_checkpoint.pt` and verifies its size and SHA-256. Wait for all parts to finish uploading. Uploading the full 823 MB checkpoint through the VS Code extension can exceed its string size limit. In browser Colab, create a `data` folder under `/content` in the Files pane and upload the checkpoint there.

Upload the complete updated `src/` folder separately. The historical CSV logs, MATLAB files, PDFs, and local `.venv` are not needed. Re-upload source files after changing them locally, and re-upload the checkpoint when the Colab runtime is replaced. The cloud copy is separate from your Mac.

Run cells in order with their ▶ buttons. The same notebook also works with a local Python kernel if the required packages and IPython are installed.

In [ ]:
from pathlib import Path
import importlib.util
import platform
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
print("Python:", sys.version.split()[0])
print("Machine:", platform.platform())
print("Runtime:", "Colab" if IN_COLAB else "local / other Jupyter")
print("Working directory:", Path.cwd())
# In Colab, paths like /Users/josef/Desktop/ASML are not available.

## Optional: verify GPU access before running the GP

On a GPU runtime, `nvidia-smi` should show the assigned NVIDIA GPU. If PyTorch is installed and CUDA is available, the next cell also runs a tiny matrix multiplication on the GPU and prints its device. This verifies actual GPU computation independently of the CPU-based GP.

A GPU name plus `cuda:0` and `finite result: True` confirms that check passed. If no GPU is available, you can still run the entire baseline experiment on CPU. Free GPU access depends on availability and account usage. Selecting a GPU runtime alone does not make scikit-learn use it. Remove/disconnect the GPU server after saving your results.

In [ ]:
import importlib.util
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=True)
else:
    print("No NVIDIA GPU detected. Select a GPU Colab runtime to test GPU access.")

if importlib.util.find_spec("torch") is not None:
    import torch
    if torch.cuda.is_available():
        with torch.no_grad():
            a = torch.randn((512, 512), device="cuda")
            product = a @ a.T
            finite = bool(torch.isfinite(product).all().item())
        torch.cuda.synchronize()
        print("GPU:", torch.cuda.get_device_name(0))
        print("Matrix multiplication device:", product.device, "| finite result:", finite)
        assert product.is_cuda and finite
        del a, product
    else:
        print("PyTorch is installed, but CUDA is unavailable on this runtime.")
else:
    print("PyTorch is not installed; nvidia-smi above checks GPU visibility only.")
print("The baseline Gaussian process below uses CPU, regardless of this GPU check.")

## 1. Find the uploaded code and prepare dependencies

The local `requirements.txt` records the original environment's exact versions. For this first cloud compatibility test, use the six required scientific libraries available to the Colab runtime; the runner records their actual versions. This does not guarantee numerical agreement across different package versions or platforms.

If the next cell cannot find the code, upload `src/` and run it again. Installing packages happens on the selected runtime, not your Mac's `.venv`.

In [ ]:
from pathlib import Path
import importlib.util
import sys

# Detect the runtime here too, so setup works after a kernel restart.
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False

import subprocess

required_files = ["run_optimization.py", "example_bayesopt.py", "srcmdl.py", "bayesopt/optimization.py", "bayesopt/models/gp.py", "bayesopt/models/kumo.py", "bayesopt/models/yield_models.py"]
candidates = [Path("/content"), Path("/content/ASML"), Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = None
SOURCE_DIR = None
for candidate in candidates:
    for source in (candidate / "src", candidate):
        if all((source / name).is_file() for name in required_files):
            PROJECT_ROOT = candidate
            SOURCE_DIR = source
            break
    if SOURCE_DIR is not None:
        break
if SOURCE_DIR is None:
    raise FileNotFoundError(
        "Upload the src folder to Colab first (VS Code: right-click src > Upload to Colab). "
        "In browser Colab upload the complete src folder, including bayesopt, and preserve its structure."
    )

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "numpy", "scipy", "matplotlib", "scikit-learn", "statsmodels", "pandas"], check=True)
else:
    print("Using the local kernel's installed dependencies; no packages were installed.")
print("Source files:", SOURCE_DIR)
print("Results will be saved under:", PROJECT_ROOT / "results")

## 2. Choose the experiment

The defaults run the original pretrained Kumo Large model, without a fine-tuned checkpoint, with the same test configuration: 200 measurements, 1,000 candidates, design/source seeds 0, all history retained, all five Kumo response predictors, four estimators, and batches of 256 on CUDA. The first 27 measurements initialize the optimizer.

The pretrained comparison uses `KUMO_CHECKPOINT = None`; no checkpoint upload is needed. Set `EUV_MODEL = "gp"` and `KUMO_CHECKPOINT = None` for the GP baseline. A supplied checkpoint is never silently replaced with pretrained weights.

To repeat the fine-tuned comparison later, set `KUMO_CHECKPOINT` to `"data/lr1e-05_T1000_tcont128_tquery512_vcont256_vquery82_checkpoint.pt"` and upload `data/checkpoint_upload_parts` if the checkpoint is no longer on the runtime. The preparation cell reassembles and verifies the parts. An explicit absolute checkpoint path on the selected runtime is also accepted.

The supplied email identifies 3800 training data and 3400 validation data; the target transforms have not been established. This comparison preserves the existing EUV residual and logit-yield transforms. Yield remains a soft penalty; Kumo intervals are not calibrated GP uncertainty.


In [ ]:
N_MEASUREMENTS = 200
N_CANDIDATES = 1000
DESIGN_SEED = 0
SOURCE_SEED = 0
FORGET_INITIAL = 0  # 0 retains all measured history; previous default was 90.
DY_MODEL = 1
COST_OPTION = 2  # Existing logit yield penalty; not a hard yield constraint.
EUV_MODEL = "kumo"
SURROGATE_SCOPE = "all"  # "all" or "euv"; applies when EUV_MODEL is "kumo".
KUMO_SIZE = "large"
KUMO_DEVICE = "cuda"  # Requires a CUDA-enabled Colab GPU runtime.
KUMO_ESTIMATORS = 4
KUMO_BATCH_SIZE = 256
KUMO_CHECKPOINT = None  # Original pretrained Kumo Large; no fine-tuned weights.
# To repeat the fine-tuned run, set this to:
# "data/lr1e-05_T1000_tcont128_tquery512_vcont256_vquery82_checkpoint.pt"


## 3. Prepare the checkpoint and run

For the default pretrained run, rerun the configuration cell above so `KUMO_CHECKPOINT = None` takes effect. No checkpoint upload is needed. The preparation cell below confirms that no custom checkpoint is selected; then run the experiment cell immediately below it.

For an optional fine-tuned run, set a checkpoint path in the configuration cell and upload `data/checkpoint_upload_parts` if needed, then run preparation and wait for **Checkpoint ready**.

Each experiment gets a new results folder. Pretrained runs use the prefix `colab_test_pretrained_`; previous fine-tuned results remain in their own folder. Detailed measurement progress is written to `run.log`. The same simulator, settings, and measurement budget are used as before. Save the completed fine-tuned run before starting the next experiment, since `RUN_DIR` will then refer to the new run.


In [ ]:
# Locate the checkpoint uploaded from this computer to the selected runtime.
from pathlib import Path

if KUMO_CHECKPOINT is None:
    print("No custom checkpoint selected; using the configured baseline.")
else:
    if EUV_MODEL != "kumo":
        raise ValueError("Set KUMO_CHECKPOINT = None for a GP baseline.")
    requested = Path(KUMO_CHECKPOINT).expanduser()
    candidates = ([requested] if requested.is_absolute() else
                  [PROJECT_ROOT / requested, Path.cwd() / requested,
                   Path("/content") / requested, Path("/content/ASML") / requested])
    selected = next((p.resolve() for p in candidates if p.is_file()), None)
    if selected is None and IN_COLAB and not requested.is_absolute():
        # Reassemble small uploads to avoid the extension's large-file limit.
        import hashlib
        import json

        manifests = sorted(Path("/content").glob("checkpoint_upload_parts/manifest.json"))
        manifests += sorted(Path("/content").glob("*/checkpoint_upload_parts/manifest.json"))
        manifests += sorted(Path("/content").glob("*/*/checkpoint_upload_parts/manifest.json"))
        for manifest_path in manifests:
            manifest = json.loads(manifest_path.read_text())
            if manifest["filename"] != requested.name:
                continue
            part_names = manifest["parts"]
            if not part_names or any(Path(name).name != name for name in part_names):
                raise ValueError(f"Invalid checkpoint parts manifest: {manifest_path}")
            missing = [name for name in part_names if not (manifest_path.parent / name).is_file()]
            if missing:
                raise FileNotFoundError(
                    "Checkpoint upload is incomplete. Upload these missing parts and rerun: "
                    + ", ".join(missing)
                )
            destination = Path("/content/data") / requested.name
            destination.parent.mkdir(parents=True, exist_ok=True)
            temporary = destination.with_suffix(".pt.partial")
            digest = hashlib.sha256()
            print("Reassembling checkpoint from uploaded parts...", flush=True)
            with temporary.open("wb") as output:
                for name in part_names:
                    with (manifest_path.parent / name).open("rb") as source:
                        while chunk := source.read(8 * 1024 * 1024):
                            output.write(chunk)
                            digest.update(chunk)
            if (temporary.stat().st_size != manifest["size"]
                    or digest.hexdigest() != manifest["sha256"]):
                temporary.unlink()
                raise ValueError("Checkpoint verification failed. Re-upload the parts and rerun.")
            temporary.replace(destination)
            selected = destination.resolve()
            print("Checkpoint size and SHA-256 verified.", flush=True)
            break
    if selected is None and IN_COLAB and not requested.is_absolute():
        # Uploading an individual file may put it directly under /content.
        runtime_checkpoints = sorted({
            p for depth in range(4)
            for p in Path("/content").glob("*/" * depth + "*.pt") if p.is_file()
        })
        matches = [p for p in runtime_checkpoints if p.name == requested.name]
        if len(matches) == 1:
            selected = matches[0].resolve()
        elif len(matches) > 1:
            raise ValueError(
                "Multiple matching checkpoints found. Set KUMO_CHECKPOINT to one "
                "absolute runtime path: " + ", ".join(str(p) for p in matches)
            )
        if selected is None:
            print("Checkpoint files found on the Colab runtime (up to 3 folders deep):", flush=True)
            for p in runtime_checkpoints:
                print(f"  {p} ({p.stat().st_size / 1024**2:.1f} MB)", flush=True)
            if not runtime_checkpoints:
                print("  No .pt files found. The checkpoint upload may not have finished, "
                      "or may have gone to a different Colab server.", flush=True)
    if selected is None:
        locations = ", ".join(str(p) for p in candidates)
        raise FileNotFoundError(
            f"Checkpoint not found. Checked: {locations}. "
            "In VS Code, right-click data/checkpoint_upload_parts > Upload to Colab "
            "and select the SAME server as this notebook. Then "
            "wait for the upload to finish, then rerun this cell. "
            "In browser Colab, upload the checkpoint into /content/data/ using "
            "the Files pane. Your Mac's files are not automatically available "
            "on the Colab runtime. For another location, set KUMO_CHECKPOINT "
            "to the checkpoint's path on the selected runtime."
        )
    if selected.stat().st_size == 0:
        raise ValueError(f"Checkpoint is empty: {selected}. Wait for the upload to finish.")

    KUMO_CHECKPOINT = str(selected)
    print(f"Checkpoint ready: {KUMO_CHECKPOINT} ({selected.stat().st_size / 1024**2:.1f} MB)")
    print("Run the next cell to start the experiment.")


In [ ]:
from pathlib import Path
import subprocess
import sys
from datetime import datetime, timezone
import os

if EUV_MODEL == "kumo" and IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "structured-data-models==0.2.1"], check=True)

checkpoint_path = None
if KUMO_CHECKPOINT is not None:
    if EUV_MODEL != "kumo":
        raise ValueError('Set KUMO_CHECKPOINT = None for a GP run.')
    requested = Path(KUMO_CHECKPOINT).expanduser()
    checkpoint_candidates = ([requested] if requested.is_absolute() else
                             [PROJECT_ROOT / requested, Path.cwd() / requested,
                              Path("/content") / requested,
                              Path("/content/ASML") / requested])
    checkpoint_path = next((p.resolve() for p in checkpoint_candidates if p.is_file()), None)
    if checkpoint_path is None:
        raise FileNotFoundError(
            f"Checkpoint not found: {KUMO_CHECKPOINT}. Upload it to the runtime's data folder "
            "alongside src, or set KUMO_CHECKPOINT to its runtime path. "
            "Set None only to run the pretrained comparison."
        )

weights_label = "finetuned" if checkpoint_path else ("pretrained" if EUV_MODEL == "kumo" else "gp")
run_name = "colab_test_" + weights_label + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = PROJECT_ROOT / "results" / run_name
runtime_env = os.environ.copy()
runtime_env.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "asml-colab-matplotlib"))
command = [sys.executable, str(SOURCE_DIR / "run_optimization.py"),
           "--n", str(N_MEASUREMENTS), "--n-test", str(N_CANDIDATES),
           "--seed", str(DESIGN_SEED), "--source-seed", str(SOURCE_SEED),
           "--warmup", str(FORGET_INITIAL),
           "--surrogate-scope", SURROGATE_SCOPE,
           "--dy-model", str(DY_MODEL), "--cost-option", str(COST_OPTION),
           "--euv-model", EUV_MODEL, "--kumo-size", KUMO_SIZE,
           "--kumo-device", KUMO_DEVICE, "--kumo-estimators", str(KUMO_ESTIMATORS),
           "--kumo-batch-size", str(KUMO_BATCH_SIZE),
           "--output", str(RUN_DIR)]
if checkpoint_path is not None:
    command.extend(["--kumo-checkpoint", str(checkpoint_path)])
print("Weights:", weights_label, "| checkpoint:", checkpoint_path)
print("Running", N_MEASUREMENTS, "measurements with EUV model:", EUV_MODEL, flush=True)
subprocess.run(command, cwd=PROJECT_ROOT, env=runtime_env, check=True)
print("Completed:", RUN_DIR)

## 4. Inspect the plot and candidate recommendation

Each measurement is one tested set of nine settings. The plot's dotted line marks the first optimizer-selected measurement. The table below selects the highest measured EUV among points meeting the measured shape limits and **underlying simulated yield ≥99.5%**.

Underlying yield is available only in the simulator and is used here for diagnosis, not to train the optimizer or choose its next measurement. In a physical experiment, repeat measurements and a statistical acceptance rule would be needed. If no point qualifies, this notebook reports that rather than recommending an unqualified point.

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

summary = json.loads((RUN_DIR / "summary.json").read_text())
measurements = pd.read_csv(RUN_DIR / "measurements.csv")
display(Image(filename=str(RUN_DIR / "overview.png"), width=1000))
print("Optimizer-selected measurements:", summary["optimizer_selected_measurements"])
print("Shape + underlying yield feasible:", summary["shape_and_actual_yield_feasible_measurements"])
qualified = measurements.loc[measurements["shape_and_actual_yield_feasible"]]
if qualified.empty:
    print("No tested setting met the shape and underlying 99.5% yield requirements.")
else:
    winner = qualified.loc[[qualified["euv_mJ"].idxmax()]]
    display(winner[["measurement", "x", "y", "z", "e", "d", "p", "k", "b",
                    "dose_target_mJ", "euv_mJ", "observed_yield", "actual_yield_simulation_only"]].T)
print("Response models:", summary["response_models"])
print("Kumo actually used:", summary.get("kumo_used", False))
print("Recorded package versions:", summary["packages"])
print("Kumo weights:", summary.get("kumo_weights", "not applicable"))
print("Checkpoint SHA-256:", summary.get("kumo_checkpoint_sha256", "not applicable"))


## 5. Keep the results after the Colab session ends

Your notebook stays saved locally in VS Code, but files created on the Colab runtime do not automatically appear in your Mac's `results/` folder. Colab runtimes are temporary.

This cell packages the results as a ZIP and prints its runtime path. In VS Code, download that ZIP from the Colab extension's **Contents** view and save it in this project's local `results/` folder before disconnecting.

In browser Colab, download the ZIP from the Files pane, or set `DOWNLOAD_IN_BROWSER = True` to trigger a direct download. Keep that switch false inside VS Code. Save the notebook with Cmd+S as well.


In [ ]:
from pathlib import Path
import shutil

DOWNLOAD_IN_BROWSER = False  # Set True only in browser Colab.
archive = Path(shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR))
print("Results archive:", archive)
if DOWNLOAD_IN_BROWSER:
    if not IN_COLAB:
        raise RuntimeError("Use the local archive path above outside Colab.")
    from google.colab import files
    files.download(str(archive))
elif IN_COLAB:
    print("Download this ZIP from the Colab Contents view in VS Code or the browser Files pane.")
    print("Save it to your computer before disconnecting; runtime storage is temporary.")


## 6. Optional: back up the completed run to Google Drive

Run only the next cell to save the experiment that already finished. Keep the same Colab runtime connected; do not rerun the experiment or use Run All. This uses the existing `RUN_DIR`, creates a ZIP, and copies it to **My Drive → ASML → results**. The model checkpoint still comes from uploaded runtime files.

Approve the Google connection prompt if asked. If mounting fails again with `FAILED_PRECONDITION`, the ZIP remains on the runtime for direct download using section 5. If the kernel lost `RUN_DIR`, set `SAVE_RUN_DIR` below to the completed run's `/content/results/...` folder.


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
from google.colab import drive

# Leave this value as-is to save the experiment that just ran.
# If needed, replace it with the completed run's absolute runtime folder path.
SAVE_RUN_DIR = globals().get("RUN_DIR")
if SAVE_RUN_DIR is None:
    available = sorted(Path("/content/results").glob("*/summary.json"))
    print("Completed run folders:")
    for path in available:
        print(path.parent)
    raise ValueError("Set SAVE_RUN_DIR above to the completed run folder, then rerun this cell.")
save_run_dir = Path(SAVE_RUN_DIR)
if not (save_run_dir / "summary.json").is_file():
    raise FileNotFoundError(f"Completed summary not found in {save_run_dir}. Check SAVE_RUN_DIR.")
saved_summary = json.loads((save_run_dir / "summary.json").read_text())
print("Saving completed run:", save_run_dir, flush=True)
print("Run configuration:", saved_summary.get("configuration", {}), flush=True)
results_archive = Path(shutil.make_archive(str(save_run_dir), "zip", root_dir=save_run_dir))
print("Runtime ZIP ready:", results_archive, flush=True)

if not Path("/content/drive/MyDrive").is_dir():
    try:
        drive.mount("/content/drive")
    except Exception as exc:
        raise RuntimeError(
            "Drive connection failed. Your completed experiment and ZIP are still on "
            "the runtime. Download the ZIP printed above using section 5 before "
            "disconnecting. You do not need to rerun the experiment."
        ) from exc
backup_dir = Path("/content/drive/MyDrive/ASML/results")
backup_dir.mkdir(parents=True, exist_ok=True)
backup_path = backup_dir / results_archive.name
temporary_backup = backup_path.with_suffix(".zip.partial")
shutil.copyfile(results_archive, temporary_backup)
with results_archive.open("rb") as source, temporary_backup.open("rb") as copied:
    if hashlib.file_digest(source, "sha256").digest() != hashlib.file_digest(copied, "sha256").digest():
        temporary_backup.unlink()
        raise IOError("Drive ZIP verification failed. Rerun this backup cell.")
temporary_backup.replace(backup_path)
print("Saved and SHA-256 verified:", backup_path)
print("Find it at drive.google.com > My Drive > ASML > results")


## What counts as a successful first test?

- The runtime information confirms Colab execution.
- The experiment finishes and the plot appears.
- `optimizer_selected_measurements` is greater than zero.
- The results ZIP is downloaded to your computer.

This demonstrates the notebook/cloud workflow. It does not establish model quality, MATLAB agreement, or a hard yield constraint. The next algorithm change is to express maximizing EUV subject to the yield threshold explicitly.

### Official setup references

- [Google Colab VS Code extension](https://marketplace.visualstudio.com/items?itemName=google.colab)
- [Google's VS Code guide: connections and file upload](https://github.com/googlecolab/colab-vscode/wiki/User-Guide)
- [Colab FAQ: free resources and temporary runtimes](https://research.google.com/colaboratory/faq.html)